# Watershed data science

Exploratory analysis of the USGS **WBD** (Watershed Boundary Dataset) HUC polygons that
back the Hydrographic Vector Art Generator, joined to **NHDPlus HR** mean-annual discharge.

This notebook lives in `notebooks/` (not `src/` or `tests/`), so — like the `tools/` scripts —
it may import the heavy GIS stack (`geopandas`/`pyogrio`) and read the real datasets directly.
The offline-suite discipline only applies to `src/` and `tests/`.

**Kernel:** select **Python (hydro-art)** (the project `.venv`).

Data read from local `datasets/`:
- WBD GDBs: `datasets/wbd/<HU2>/WBD_<HU2>_HU2_GDB.gdb` — layers `WBDHU2/4/6/8/10/12/14/16`
- NHDPlus HR GDBs: `datasets/nhdplus_hr/<HUC4>/NHDPLUS_H_<HUC4>_HU4_GDB.gdb` — layer `NHDPlusEROMMA` (`QAMA` = mean-annual cfs)

## 1. Setup

In [ ]:
import glob
import sys
from pathlib import Path

# Make the repo root importable so we can reuse project constants (e.g. src.crs).
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.crs import INTERNAL_CRS  # single source of truth: EPSG:5070 (metric)

pd.set_option("display.max_columns", 40)
print(f"repo root : {REPO}")
print(f"internal CRS: {INTERNAL_CRS}")

## 2. Discover the local WBD data

In [ ]:
import pyogrio

WBD_GLOB = str(REPO / "datasets" / "wbd" / "**" / "*.gdb")
wbd_gdbs = sorted(glob.glob(WBD_GLOB, recursive=True))
print(f"Found {len(wbd_gdbs)} WBD GDB(s):")
for p in wbd_gdbs:
    print("  ", Path(p).relative_to(REPO))

# HUC levels available in the first GDB:
layers = [name for name, _ in pyogrio.list_layers(wbd_gdbs[0])]
print("\nWBD layers:", [l for l in layers if l.startswith("WBDHU")])

## 3. Load HUC polygons

Set `HUC_LEVEL` to any available level (`WBDHU4` is a good starting granularity — the basins the
art pipeline colors by). We concatenate every WBD GDB, reproject to the project's metric CRS
(EPSG:5070), and compute a clean `area_km2` from the projected geometry.

In [ ]:
HUC_LEVEL = "WBDHU4"       # try WBDHU8 / WBDHU12 for finer basins
HUC_COL = HUC_LEVEL.replace("WBDHU", "huc").lower()  # 'huc4'

frames = []
for gdb in wbd_gdbs:
    gdf = gpd.read_file(gdb, layer=HUC_LEVEL, columns=[HUC_COL, "name", "states", "areasqkm"])
    frames.append(gdf)

huc = pd.concat(frames, ignore_index=True)
huc = gpd.GeoDataFrame(huc, geometry="geometry", crs=frames[0].crs)
huc = huc.to_crs(INTERNAL_CRS)
huc["area_km2"] = huc.geometry.area / 1e6  # projected area, sanity-check vs 'areasqkm'
huc = huc.drop_duplicates(subset=HUC_COL).reset_index(drop=True)

print(f"{len(huc)} {HUC_LEVEL} polygons, CRS = {huc.crs.to_string()}")
huc.head()

## 4. Exploratory summary

In [ ]:
display(huc["area_km2"].describe().to_frame("area_km2"))

# Basins by state (the WBD 'states' field is a comma-separated list).
by_state = (
    huc.assign(state=huc["states"].str.split(","))
    .explode("state")
    .groupby("state")
    .agg(n_basins=(HUC_COL, "nunique"), total_km2=("area_km2", "sum"))
    .sort_values("total_km2", ascending=False)
)
display(by_state)

print("Largest basins:")
huc.nlargest(5, "area_km2")[[HUC_COL, "name", "states", "area_km2"]]

## 5. Map the basins

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

huc.plot(
    column="area_km2", cmap="viridis", legend=True,
    edgecolor="white", linewidth=0.3, ax=ax1,
)
ax1.set_title(f"{HUC_LEVEL} basins by area (km\u00b2)")
ax1.set_axis_off()

huc["area_km2"].plot.hist(bins=20, ax=ax2, color="#2a9d8f")
ax2.set_title("Basin area distribution")
ax2.set_xlabel("area (km\u00b2)")

plt.tight_layout()
plt.show()

## 6. Join NHDPlus discharge to each basin

A worked example of a spatial/attribute join: each NHDPlus HR GDB is named by its HUC4, so we
aggregate mean-annual discharge (`QAMA`, cfs) per HUC4 from the `NHDPlusEROMMA` table and merge it
onto the WBD polygons. (For sub-HUC4 levels you'd instead join reaches to polygons spatially.)

In [ ]:
nhd_gdbs = sorted(glob.glob(str(REPO / "datasets" / "nhdplus_hr" / "*" / "*.gdb")))

rows = []
for gdb in nhd_gdbs:
    huc4 = Path(gdb).parent.name  # e.g. '1709'
    erom = gpd.read_file(gdb, layer="NHDPlusEROMMA", columns=["QAMA"], read_geometry=False)
    q = pd.to_numeric(erom["QAMA"], errors="coerce")
    q = q[q > 0]  # drop nodata / non-positive
    rows.append({"huc4": huc4, "reaches": int(q.size),
                 "q_max_cfs": float(q.max()), "q_total_cfs": float(q.sum())})

discharge = pd.DataFrame(rows)
print(f"Aggregated discharge for {len(discharge)} HUC4 basin(s)")
discharge.sort_values("q_max_cfs", ascending=False).head()

In [ ]:
# Merge onto HUC4 polygons (works when HUC_LEVEL == 'WBDHU4'; for finer levels
# derive the 4-digit prefix first).
key = huc[HUC_COL].str[:4]
joined = huc.assign(huc4=key).merge(discharge, on="huc4", how="left")
joined = gpd.GeoDataFrame(joined, geometry="geometry", crs=huc.crs)

ax = joined.plot(
    column="q_max_cfs", cmap="plasma", legend=True,
    edgecolor="white", linewidth=0.3, missing_kwds={"color": "lightgrey"},
    figsize=(8, 8),
)
ax.set_title("Peak mean-annual reach discharge per basin (cfs)")
ax.set_axis_off()
plt.show()

joined[["huc4", "name", "area_km2", "reaches", "q_max_cfs"]].sort_values(
    "q_max_cfs", ascending=False
).head(10)

## 7. Cluster basins with scikit-learn

Group HUC4 basins into "types" from a few features — size, peak discharge, and a drainage-density
proxy (`reaches / area_km2`). We standardize the features (KMeans is scale-sensitive) and fix
`random_state` so the labels are deterministic, in keeping with the project's reproducibility ethos.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Feature engineering from the joined basins (drop basins with no discharge match).
feat = joined.dropna(subset=["q_max_cfs", "reaches"]).copy()
feat["reach_density"] = feat["reaches"] / feat["area_km2"]  # drainage-density proxy
feat["mean_q_per_reach"] = feat["q_total_cfs"] / feat["reaches"]

FEATURES = ["area_km2", "q_max_cfs", "reach_density", "mean_q_per_reach"]
X = StandardScaler().fit_transform(feat[FEATURES])

K = min(4, len(feat))  # guard against fewer basins than clusters
km = KMeans(n_clusters=K, random_state=0, n_init=10)
feat["cluster"] = km.fit_predict(X).astype(str)

# Per-cluster profile (means in original units).
profile = feat.groupby("cluster")[FEATURES].mean().round(1)
profile["n_basins"] = feat.groupby("cluster").size()
display(profile)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
feat.plot(column="cluster", categorical=True, legend=True, cmap="Set2",
          edgecolor="white", linewidth=0.3, ax=ax1)
ax1.set_title(f"Basin clusters (k={K})")
ax1.set_axis_off()

for c, grp in feat.groupby("cluster"):
    ax2.scatter(grp["area_km2"], grp["q_max_cfs"], label=f"cluster {c}", s=60)
ax2.set(xlabel="area (km\u00b2)", ylabel="peak discharge (cfs)", title="Area vs. peak discharge")
ax2.legend()
plt.tight_layout()
plt.show()

## Next steps

- Switch `HUC_LEVEL` to `WBDHU8`/`WBDHU12` for finer basins.
- The project's graph-based HUC grouping lives in `src/watersheds.py` (`group_segments_by_huc`); the
  monthly-flow disaggregation model is `src/monthly_flow.py` / `src/historical_flow.py` — all pure and
  reusable from here.
- `scikit-learn` is installed: e.g. cluster basins by (area, discharge, drainage density) features.
- For interactive slippy maps, `huc.explore()` works (GeoPandas + folium).